# 🪔 Akshara-Drishti — Training Notebook
**AI-Powered Indic Script Intelligence System**

Classifies inscription images into: `Brahmi`, `Devanagari`, `Tamil`

---
> ⚠️ **Run Cell 1 first, then restart the runtime before continuing.**

In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES  (Run once → restart runtime)
# ============================================================
# FIX: was buried at bottom of file; must run before any import
!pip install -q tensorflow==2.15.0 opencv-python-headless pillow matplotlib

In [ ]:
# ============================================================
# CELL 2 — ALL IMPORTS  (single consolidated block)
# ============================================================
# FIX: removed duplicate import blocks; added all missing imports here
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# ============================================================
# CELL 3 — MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CELL 4 — PATHS & HYPERPARAMETERS
# ============================================================
BASE_PATH    = "/content/drive/MyDrive/AKSHARA_DHRISTI_SOFTSKILLS"
DATASET_PATH = BASE_PATH + "/dataset"
MODEL_PATH   = BASE_PATH + "/best_model.keras"   # checkpoint target
FINAL_PATH   = BASE_PATH + "/script_model.keras"  # final saved model

IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS_S1  = 20   # Stage 1 — frozen base
EPOCHS_S2  = 10   # Stage 2 — fine-tune
SEED       = 123

In [ ]:
# ============================================================
# CELL 5 — CLEAN & RENAME DATASET
# ============================================================
def clean_and_rename(folder_path, prefix):
    """Rename all valid images to a consistent numbered format."""
    if not os.path.exists(folder_path):
        print(f"⚠️  Folder not found: {folder_path}")
        return

    files = sorted(os.listdir(folder_path))  # sorted for determinism
    count = 1
    renamed = 0

    for file in files:
        if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        old_path = os.path.join(folder_path, file)
        new_name = f"{prefix}_{count:05d}.jpg"
        new_path = os.path.join(folder_path, new_name)

        # Avoid overwriting an existing file with a different name
        while os.path.exists(new_path) and new_path != old_path:
            count += 1
            new_name = f"{prefix}_{count:05d}.jpg"
            new_path = os.path.join(folder_path, new_name)

        if old_path != new_path:
            os.rename(old_path, new_path)
            renamed += 1
        count += 1

    print(f"✅ Renamed {renamed} files in: {prefix}/")

clean_and_rename(DATASET_PATH + "/brahmi",      "brahmi")
clean_and_rename(DATASET_PATH + "/devanagari",  "devanagari")
clean_and_rename(DATASET_PATH + "/tamil",       "tamil")

In [ ]:
# ============================================================
# CELL 6 — REMOVE CORRUPTED IMAGES
# ============================================================
# FIX: restored from commented-out state; fixed broken indentation
def remove_bad_images(folder):
    """Delete any image that Pillow cannot open/verify."""
    if not os.path.exists(folder):
        print(f"⚠️  Folder not found: {folder}")
        return

    removed = 0
    for file in os.listdir(folder):
        if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        path = os.path.join(folder, file)
        try:
            img = Image.open(path)
            img.verify()          # catches truncated / corrupted files
        except Exception as e:
            os.remove(path)
            removed += 1
            print(f"  ❌ Removed corrupt file: {file}  ({e})")
    print(f"✅ Corruption check done — removed {removed} file(s) from {os.path.basename(folder)}/")

remove_bad_images(DATASET_PATH + "/brahmi")
remove_bad_images(DATASET_PATH + "/devanagari")
remove_bad_images(DATASET_PATH + "/tamil")

In [ ]:
# ============================================================
# CELL 7 — LOAD DATASET
# ============================================================
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f"✅ Classes found: {class_names}")
print(f"   Train batches : {len(train_ds)}")
print(f"   Val   batches : {len(val_ds)}")

In [ ]:
# ============================================================
# CELL 8 — CLAHE PREPROCESSING
# ============================================================
def apply_clahe(image):
    """Apply CLAHE contrast enhancement to a single image tensor."""
    image = image.numpy().astype(np.uint8)
    gray  = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)
    return enhanced

def tf_clahe(images, labels):
    """Batch-level CLAHE wrapper compatible with tf.data.Dataset.map."""
    images = tf.map_fn(
        lambda img: tf.py_function(apply_clahe, [img], tf.uint8),
        images,
        fn_output_signature=tf.uint8
    )
    images.set_shape((None, IMG_SIZE, IMG_SIZE, 3))
    return images, labels

In [ ]:
# ============================================================
# CELL 9 — BUILD tf.data PIPELINE
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

# Apply CLAHE
train_ds = train_ds.map(tf_clahe, num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.map(tf_clahe,   num_parallel_calls=AUTOTUNE)

# Normalise to [0, 1]
train_ds = train_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y),
                        num_parallel_calls=AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y),
                      num_parallel_calls=AUTOTUNE)

# Cache → Shuffle → Prefetch  (val doesn't need shuffle)
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

print("✅ Pipeline ready")

In [ ]:
# ============================================================
# CELL 10 — DATA AUGMENTATION BLOCK
# ============================================================
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomFlip("horizontal"),   # bonus: horizontal flip
], name="augmentation")

In [ ]:
# ============================================================
# CELL 11 — BUILD MODEL
# ============================================================
base_model = tf.keras.applications.EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False   # frozen for Stage 1

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = data_augmentation(inputs)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dense(128, activation='relu')(x)
x       = layers.Dropout(0.5)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = Model(inputs, outputs, name="AksharaDrishti")
model.summary()

In [ ]:
# ============================================================
# CELL 12 — CALLBACKS
# ============================================================
# FIX: imports moved to top; defined once, reused in both stages
checkpoint = ModelCheckpoint(
    MODEL_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True,
    verbose=1
)

callbacks = [checkpoint, early_stop]
print("✅ Callbacks ready")

In [ ]:
# ============================================================
# CELL 13 — STAGE 1: COMPILE & TRAIN  (frozen base)
# ============================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# FIX: callbacks= was missing from the original model.fit() call
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_S1,
    callbacks=callbacks          # ← was absent in original
)

In [ ]:
# ============================================================
# CELL 14 — PLOT STAGE 1 RESULTS
# ============================================================
# FIX: original only plotted accuracy; loss curve added
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train',      color='royalblue')
axes[0].plot(history.history['val_accuracy'], label='Validation', color='gold')
axes[0].set_title('Stage 1 — Accuracy',  fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train',      color='royalblue')
axes[1].plot(history.history['val_loss'], label='Validation', color='gold')
axes[1].set_title('Stage 1 — Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 15 — STAGE 2: FINE-TUNE  (unfreeze last 30 layers)
# ============================================================
base_model.trainable = True

# Keep earlier layers frozen — only fine-tune the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f"✅ Fine-tuning {trainable_count} / {len(base_model.layers)} base layers")

# Lower LR to avoid destroying pretrained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# FIX: callbacks= was missing from the fine-tune fit() call too
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_S2,
    callbacks=callbacks          # ← was absent in original
)

In [ ]:
# ============================================================
# CELL 16 — PLOT STAGE 2 RESULTS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_fine.history['accuracy'],     label='Train',      color='royalblue')
axes[0].plot(history_fine.history['val_accuracy'], label='Validation', color='gold')
axes[0].set_title('Stage 2 Fine-Tune — Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_fine.history['loss'],     label='Train',      color='royalblue')
axes[1].plot(history_fine.history['val_loss'], label='Validation', color='gold')
axes[1].set_title('Stage 2 Fine-Tune — Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 17 — SAVE FINAL MODEL
# ============================================================
model.save(FINAL_PATH)
print(f"✅ Final model saved to: {FINAL_PATH}")
print(f"✅ Best checkpoint at : {MODEL_PATH}")
print(f"\n📦 Copy script_model.keras → backend/model/ for deployment.")